# Coco Crepe — 04 Silver Sales

Transformación del Data Product de ventas consumiendo `gold.product_master`.

**Mejora aplicada:** deduplicación por llaves de negocio antes de los joins:
- `customer_id` para clientes.
- `order_id` para órdenes.
- `item_id` para detalle de órdenes.

Esto evita que registros duplicados generen doble conteo de unidades o ingresos.

In [0]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# Completa estos valores solo si la detección automática no encuentra
# exactamente un catálogo por Data Product.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

In [0]:
product_master = f"{PRODUCT_CATALOG}.gold.product_master_final"

sales_customers = f"{SALES_CATALOG}.bronze.customers_final"
sales_orders = f"{SALES_CATALOG}.bronze.orders_final"
sales_order_items = f"{SALES_CATALOG}.bronze.order_items_final"

sales_detail = f"{SALES_CATALOG}.silver.sales_detail_final"

## Diagnóstico de duplicados en Bronze

In [0]:
spark.sql(f"""
SELECT
    'customers' AS source_table,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_business_keys
FROM {sales_customers}

UNION ALL

SELECT
    'orders',
    COUNT(*),
    COUNT(DISTINCT order_id),
    COUNT(*) - COUNT(DISTINCT order_id)
FROM {sales_orders}

UNION ALL

SELECT
    'order_items',
    COUNT(*),
    COUNT(DISTINCT item_id),
    COUNT(*) - COUNT(DISTINCT item_id)
FROM {sales_order_items}
""").display()

## Construcción de Silver Sales

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {sales_detail} AS
WITH customers_ranked AS (
    SELECT
        customer_id,
        customer_name,
        district,
        registration_date,
        inserted_at,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY
                inserted_at DESC,
                registration_date DESC,
                customer_name DESC
        ) AS row_number
    FROM {sales_customers}
    WHERE customer_id IS NOT NULL
),
customers_deduplicated AS (
    SELECT
        customer_id,
        customer_name,
        district,
        registration_date
    FROM customers_ranked
    WHERE row_number = 1
),
orders_ranked AS (
    SELECT
        order_id,
        order_date,
        customer_id,
        total_amount,
        payment_method,
        inserted_at,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY
                inserted_at DESC,
                order_date DESC,
                total_amount DESC
        ) AS row_number
    FROM {sales_orders}
    WHERE order_id IS NOT NULL
),
orders_deduplicated AS (
    SELECT
        order_id,
        order_date,
        customer_id,
        total_amount,
        payment_method
    FROM orders_ranked
    WHERE row_number = 1
),
order_items_ranked AS (
    SELECT
        item_id,
        order_id,
        product_id,
        quantity,
        unit_price,
        inserted_at,
        ROW_NUMBER() OVER (
            PARTITION BY item_id
            ORDER BY
                inserted_at DESC,
                order_id DESC,
                product_id DESC,
                quantity DESC,
                unit_price DESC
        ) AS row_number
    FROM {sales_order_items}
    WHERE item_id IS NOT NULL
),
order_items_deduplicated AS (
    SELECT
        item_id,
        order_id,
        product_id,
        quantity,
        unit_price
    FROM order_items_ranked
    WHERE row_number = 1
)
SELECT
    o.order_id,
    oi.item_id,
    CAST(o.order_date AS DATE) AS order_date,
    o.customer_id,
    INITCAP(TRIM(c.customer_name)) AS customer_name,
    UPPER(TRIM(c.district)) AS district,
    oi.product_id,
    p.product_name,
    p.category,
    CAST(oi.quantity AS INT) AS quantity,
    ROUND(CAST(oi.unit_price AS DOUBLE), 2) AS unit_price,
    ROUND(
        CAST(oi.quantity AS DOUBLE) *
        CAST(oi.unit_price AS DOUBLE),
        2
    ) AS line_total,
    UPPER(TRIM(o.payment_method)) AS payment_method,
    current_timestamp() AS transformed_at
FROM orders_deduplicated o
INNER JOIN order_items_deduplicated oi
    ON o.order_id = oi.order_id
INNER JOIN {product_master} p
    ON oi.product_id = p.product_id
LEFT JOIN customers_deduplicated c
    ON o.customer_id = c.customer_id
WHERE oi.quantity > 0
  AND oi.unit_price > 0
  AND p.is_active = TRUE
""")

## Validaciones de Silver Sales

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS record_count,
    COUNT(DISTINCT item_id) AS distinct_items,
    COUNT(*) - COUNT(DISTINCT item_id) AS duplicate_items,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_ids,
    SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END) AS null_item_ids,
    SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantities,
    SUM(CASE WHEN unit_price <= 0 THEN 1 ELSE 0 END) AS invalid_prices,
    SUM(
        CASE
            WHEN ABS(line_total - ROUND(quantity * unit_price, 2)) > 0.01
            THEN 1 ELSE 0
        END
    ) AS incorrect_line_totals
FROM {sales_detail}
""").display()

spark.sql(f"SELECT * FROM {sales_detail} ORDER BY order_date, order_id LIMIT 20").display()